In [ ]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 20.7 MB/s eta 0:00:00


In [1]:
!ls -la


total 16
drwxr-xr-x 1 root root 4096 Mar 14 13:32 .
drwxr-xr-x 1 root root 4096 Mar 18 00:30 ..
drwxr-xr-x 4 root root 4096 Mar 14 13:32 .config
drwxr-xr-x 1 root root 4096 Mar 14 13:32 sample_data


In [ ]:
import os
import json
import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering, AutoTokenizer, AutoModel, LongformerForQuestionAnswering
from transformers import T5Tokenizer, T5ForConditionalGeneration

def return_json_text(json_file):
    with open(json_file) as f:
        data = json.load(f)
        text = data['text']
    return text

def document_list(folder_name):
    documents = []
    for file_name in os.listdir(folder_name):
        file_path = os.path.join(folder_name, file_name)
        if file_path.endswith(".json"):
            documents.append(return_json_text(file_path))
        elif file_path.endswith(".txt"):
            with open(file_path, "r") as f:
                documents.append(f.read())
    return documents


# SAFETY AND NO MULTIPROCESSING
os.environ["OMP_NUM_THREADS"] = "1"
torch.set_num_threads(1)
faiss.omp_set_num_threads(1)

'''*************************'''
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
'''*************************'''

# DOCUMENT LIST, replace with function output for the json txts
data_folder_path = "/content/drive/MyDrive/11711 data/gold_documents"
documents = document_list(data_folder_path)

# use embed_model to embed all documeents
passage_embeddings = embed_model.encode(documents, normalize_embeddings=True)

# force embeddings to match FAISS index dimension
dimension = passage_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(passage_embeddings)
print(f"Indexed {index.ntotal} documents with FAISS.")

def retrieve_documents(query, top_k=2):
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_embedding, top_k)

    results = [(documents[idx], distances[0][i]) for i, idx in enumerate(indices[0])]
    return results

'''*************************'''

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
qa_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

'''tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
qa_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")'''

'''tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-large-4096-finetuned-triviaqa")
qa_model = LongformerForQuestionAnswering.from_pretrained("allenai/longformer-large-4096-finetuned-triviaqa")'''

'''qa_model = DistilBertForQuestionAnswering.from_pretrained("distilbert/distilbert-base-cased-distilled-squad")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert/distilbert-base-cased-distilled-squad")'''
'''*************************'''

'''
def retrieve_and_answer(question, top_k=1):
    """Retrieve relevant document and answer the question using DistilBERT."""

    # Retrieve relevant document from FAISS
    query_embedding = embed_model.encode([question], normalize_embeddings=True)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_doc = documents[indices[0][0]]  # Select top-ranked document

    # Combine retrieved document with the question
    context = retrieved_doc
    #print(f"\n Retrieved Context: {context}")

    # Tokenize input for QA model
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)

    # LLM OUTPUT
    with torch.no_grad():
        outputs = qa_model(**inputs)

    # answer
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end]))

    return answer
'''
def retrieve_and_answer(question, top_k=3):
    """Retrieve relevant document and answer the question using T5."""

    # query_embedding = embed_model.encode([question], normalize_embeddings=True)
    # distances, indices = index.search(query_embedding, top_k)
    query_embedding = embed_model.encode([question], normalize_embeddings=True)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_doc = documents[indices[0][0]]

    # Only use the first 900 tokens of the document
    retrieved_doc_tokens = tokenizer.tokenize(retrieved_doc)[:6000]
    retrieved_doc = tokenizer.convert_tokens_to_string(retrieved_doc_tokens)

    multi_answer_instruction = ""
    if (" more than " in question.lower() or " both " in question.lower() or " two " in question.lower() or " three " in question.lower()):
        multi_answer_instruction = "List all answers, separated by semicolons. "

    # # Only use the first 900 tokens of the document
    # retrieved_doc_tokens = tokenizer.tokenize(retrieved_doc)[:900]
    # retrieved_doc = tokenizer.convert_tokens_to_string(retrieved_doc_tokens)


    # PRINT the name of the document
    #print(f"\n Retrieved Document: {retrieved_doc}")

    # T5 expects full context
    input_text = f"{multi_answer_instruction}Answer this question based on the context: {retrieved_doc} Question: {question}"

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True,max_length=6200)

    with torch.no_grad():
        outputs = qa_model.generate(**inputs, max_length=50)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


# QUESTIONS
questions = [
    # Factual and Never Changing Questions
    "Who founded the Carnegie Museum of Natural History?",
    "What is the official dessert of Pennsylvania?",
    "In what year was the first Carnegie International held?",
    "Where is the Andy Warhol Museum located?",
    "What is the name of the natural reserve associated with the Carnegie Museum of Natural History?",
    "What landmark structures were completed in 1970 as part of Pittsburgh’s “Renaissance I” urban renewal?",
    "What nickname was given to Pittsburgh by 1850?",
    "What was the white alone (non-Hispanic) population in 2020?" ,
    "Name the two major daily newspapers serving Pittsburgh.",
    "Name the two largest health care providers in the Pittsburgh area.",

    # Slow Changing Questions
    "How many specimens does the Carnegie Museum of Natural History hold?",
    "What is the operating budget of the Andy Warhol Museum?",
    "What are the security policies for Heinz Hall performances?",
    "Approximately how many families settled in western Pennsylvania between 1768 and 1770?",
    "What cumulative economic impact has Anthrocon generated over 11 years?",
    "What major building serves as the official seat of government in Pittsburgh?",
    "Name one of the water service providers for Pittsburgh",




    # Fast Changing Questions (Future Events, Concerts, and Temporary Exhibitions)
    "What concerts are scheduled at PPG Paints Arena in April 2025?",
    "When is Pittsburgh Restaurant Week 2025 happening?",
    "What vendors will be featured at Picklesburgh this year?",
    "When is the next Pittsburgh Symphony Orchestra rush ticket program?",
    "What were Pittsburgh's population figures in 2010 and 2020?" ,
    "When were the Parking Tax Regulations most recently revised?",
    "What is the name of the historic urban renewal project that generates over 3.5 million visitors a year?",
    "Who is the first African-American mayor of Pittsburgh?",
    "What is the average wait time at a public transit stop in Pittsburgh?",

    # False Premise Questions (to test RAG's correction ability)
    "Which museum in Pittsburgh was originally built as a castle?",
    "Who won the Andy Warhol Museum’s 2023 artist of the year award?",
    "When did Heinz Hall first allow audiences to bring their pets to performances?",
    "What new dinosaur species was discovered in Pittsburgh in 2024?",
    "How many concerts has the Pittsburgh Symphony Orchestra played at Carnegie Hall this year?"
]


# for question in questions:
#     print(f"\n Question: {question}")
#     answer = retrieve_and_answer(question)
#     print(f"\n Answer: {answer}")

Indexed 102 documents with FAISS.


In [ ]:
import os
import sentencepiece
from transformers import T5Tokenizer
import numpy as np


# Initialize the tokenizer
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")

# Directory containing the files
directory = '/content/drive/MyDrive/11711 data/gold_documents'

# Iterate over each file in the directory
list_of_tokens = []
for filename in os.listdir(directory):
    file_path = os.path.join(directory, filename)

    # Check if it is a file
    if os.path.isfile(file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
            tokens = tokenizer.tokenize(text)
            num_tokens = len(tokens)
            print(f"File: {filename}, Number of tokens: {num_tokens}")
            list_of_tokens.append(num_tokens)

# Distribution of token lengths
print(f"Mean number of tokens: {sum(list_of_tokens)/len(list_of_tokens)}")
print(f"Max number of tokens: {max(list_of_tokens)}")
print(f"Standard deviation of tokens: {np.std(list_of_tokens)}")

File: pghmuseums.txt, Number of tokens: 936
File: amusement_tax.txt, Number of tokens: 12347
File: pghrestaurants.txt, Number of tokens: 2252
File: pca.txt, Number of tokens: 1946
File: randyland.txt, Number of tokens: 895
File: cmuNationalAcademies.txt, Number of tokens: 1600
File: heinzhallhistory.txt, Number of tokens: 3218
File: frick.txt, Number of tokens: 1033
File: uf_reg_tax.txt, Number of tokens: 5502
File: festivals_events.txt, Number of tokens: 44343
File: historywiki_SteelCity.txt, Number of tokens: 2145
File: snswiki.txt, Number of tokens: 703
File: wiki_utilities.txt, Number of tokens: 72
File: little_italy.txt, Number of tokens: 777
File: cmawiki.txt, Number of tokens: 1068
File: trust_arts.txt, Number of tokens: 4117
File: wiki_sistercities.txt, Number of tokens: 139
File: operaabout.txt, Number of tokens: 258
File: pittsburghconcerts.txt, Number of tokens: 3405
File: wiki_healthcare.txt, Number of tokens: 954
File: benedum.txt, Number of tokens: 1214
File: wiki_intro.t

In [ ]:
import pandas as pd
questions2 = pd.read_csv("/content/drive/MyDrive/test_set.csv")
questions_list = questions2.iloc[:, 0].tolist()
print(questions_list)

['Which counties host major maple festivals near Pittsburgh?', "Which new sports team is mentioned as completing the pipeline from the Riverhounds Academy to professional women's soccer?", "Where is The Driver Era's tour scheduled to stop in Pittsburgh?", 'What can one explore with the free walking tours in Pittsburgh?', 'Which Pittsburgh festival involves a competitive pickle juice drinking contest?', 'What type of facility is the Inglis Innovation Center planning to include in its Bellevue location?', 'When is the Great American Banana Split Celebration in 2025?', 'When is Spring Carnival Weekend at CMU?', 'Which Pittsburgh food event features a competitive pickle juice drinking contest?', 'Where is the Big Nosh Jewish Food Festival taking place?', 'What type of artworks can one explore at The Andy Warhol Museum in Pittsburgh?', 'Which event celebrating nature will take place on March 22, 2025?', 'Where can one find free entertainment in downtown Pittsburgh on the first or third Mond

In [ ]:
for question in questions_list:
    print(f"\n Question: {question}")
    answer = retrieve_and_answer(question)
    print(f"\n Answer: {answer}")


 Question: Which counties host major maple festivals near Pittsburgh?

 Answer: unanswerable

 Question: Which new sports team is mentioned as completing the pipeline from the Riverhounds Academy to professional women's soccer?

 Answer: Steel City Yellow Jackets

 Question: Where is The Driver Era's tour scheduled to stop in Pittsburgh?

 Answer: UPMC Events Center

 Question: What can one explore with the free walking tours in Pittsburgh?

 Answer: Pittsburgh, Pennsylvania

 Question: Which Pittsburgh festival involves a competitive pickle juice drinking contest?

 Answer: Picklesburgh

 Question: What type of facility is the Inglis Innovation Center planning to include in its Bellevue location?

 Answer: research and technology

 Question: When is the Great American Banana Split Celebration in 2025?

 Answer: The Great American Banana Split Celebration in 2025

 Question: When is Spring Carnival Weekend at CMU?

 Answer: april 3-5, 2025

 Question: Which Pittsburgh food event featu